## 1. Installation des dépendances

In [1]:
!pip install -q pandas openpyxl sentence-transformers rank-bm25 faiss-cpu huggingface_hub unidecode


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.1 MB/s eta 0:00:00


## 2. Chargement et nettoyage des données (Google Drive ou upload local)

In [2]:
import pandas as pd
import os
import re

# --- 1. Chargement du fichier Excel : Google Drive (Colab), avec repli sur upload local ---
EXCEL_PATH = '/content/drive/MyDrive/ORIENT_IA_Dataset_Niveaux_1_a_5.xlsx'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if not os.path.exists(EXCEL_PATH):
    if ON_COLAB:
        print(f"⚠️ Fichier non trouvé à l'emplacement '{EXCEL_PATH}'.")
        from google.colab import files
        print("📤 Merci d'uploader le fichier ORIENT_IA_Dataset_Niveaux_1_a_5.xlsx :")
        uploaded = files.upload()
        EXCEL_PATH = list(uploaded.keys())[0]
    else:
        # Environnement local (Jupyter classique, hors Colab)
        EXCEL_PATH = "ORIENT_IA_Dataset_Niveaux_1_a_5.xlsx"
        if not os.path.exists(EXCEL_PATH):
            raise FileNotFoundError(
                "Fichier introuvable. Placez 'ORIENT_IA_Dataset_Niveaux_1_a_5.xlsx' "
                "dans le même dossier que ce notebook, ou modifiez EXCEL_PATH."
            )

print(f"✅ Fichier trouvé : {EXCEL_PATH}")
xls = pd.ExcelFile(EXCEL_PATH)
print(f"Feuilles disponibles ({len(xls.sheet_names)}) :", xls.sheet_names)

sheets_dict_raw = {s: pd.read_excel(xls, s) for s in xls.sheet_names}

# --- 2. Nettoyage systématique : espaces parasites dans les noms de colonnes et les valeurs texte ---
def nettoyer_dataframe(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda v: v.strip() if isinstance(v, str) else v)
            df[col] = df[col].replace(r'^\s*$', pd.NA, regex=True)
    return df

sheets_dict = {name: nettoyer_dataframe(df) for name, df in sheets_dict_raw.items()}

# --- 3. Normalisation des codes parcours (espaces, casse, sauts de ligne, coquilles connues) ---
ALIAS_CODES = {"ISAIIA": "ISAIA"}  # coquille repérée dans la feuille 'Compétences développées'

def normaliser_code(code):
    if pd.isna(code):
        return None
    c = re.sub(r'\s+', '', str(code)).upper()
    return ALIAS_CODES.get(c, c)

for _name, _df in sheets_dict.items():
    for _col in _df.columns:
        if _col.startswith("Code_Parcours"):
            _df[_col] = _df[_col].apply(normaliser_code)

# --- 4. Repérage robuste des feuilles utiles (résiste à un léger renommage des onglets) ---
def get_sheet(fragment):
    for s in sheets_dict:
        if fragment.lower() in s.lower():
            return s
    raise KeyError(f"Aucune feuille trouvée contenant '{fragment}'. Feuilles dispo : {list(sheets_dict.keys())}")

SHEET_FORMATIONS  = get_sheet("Formations_Matieres")
SHEET_DEBOUCHES   = get_sheet("Débouchés")
SHEET_RELATION    = get_sheet("Relation")
SHEET_PASSERELLES = get_sheet("passerelle")
SHEET_CONDITIONS  = get_sheet("Conditions")
SHEET_COMPETENCES = get_sheet("Compétences développées")

for sheet, df in sheets_dict.items():
    print(f"  * {sheet} : {len(df)} lignes, {len(df.columns)} colonnes")

print("\n✅ Données chargées et nettoyées. Codes parcours uniques détectés :")
print(sorted(sheets_dict[SHEET_FORMATIONS]['Code_Parcours'].dropna().unique().tolist()))


Mounted at /content/drive
✅ Fichier trouvé : /content/drive/MyDrive/ORIENT_IA_Dataset_Niveaux_1_a_5.xlsx
Feuilles disponibles (7) : ['01_Formations_Matieres', 'Débouchés pro', 'Relation(parcours, métiers, com', 'Eventuelle passerelle entre for', '00_README', 'Conditions daccès en L1', 'Compétences développées']
  * 01_Formations_Matieres : 80 lignes, 12 colonnes
  * Débouchés pro : 16 lignes, 7 colonnes
  * Relation(parcours, métiers, com : 16 lignes, 10 colonnes
  * Eventuelle passerelle entre for : 40 lignes, 9 colonnes
  * 00_README : 6 lignes, 2 colonnes
  * Conditions daccès en L1 : 16 lignes, 9 colonnes
  * Compétences développées : 16 lignes, 7 colonnes

✅ Données chargées et nettoyées. Codes parcours uniques détectés :
['AEE', 'CAA', 'DTJA', 'EMII', 'EMP', 'ESIIA', 'FIC', 'GCA', 'IAA', 'ICMP', 'IGGLIA', 'IMTICIA', 'ISAIA', 'PIP', 'TEE', 'TEH']


## 3. Construction des chunks RAG (par ligne + synthèses par parcours)

In [3]:
# Colonnes de traçabilité conservées comme métadonnées mais exclues du texte indexé
# (elles ajoutent du bruit lexical/sémantique sans aider la pertinence de la recherche)
COLONNES_METADONNEES = {"Source", "Date_Consultation", "Statut_Source", "Niveau_Confiance", "Notes_Incertaines"}


def construire_chunks_lignes(sheets_dict):
    """Transforme chaque ligne pertinente de chaque feuille en un chunk texte + métadonnées."""
    chunks = []
    for sheet_name, df in sheets_dict.items():
        if sheet_name == "00_README":
            continue

        for index, row in df.iterrows():
            if row.dropna().empty:
                continue

            code_parcours = row.get("Code_Parcours", row.get("Code_Parcours_Origine", None))
            mention = row.get("Mention", row.get("Mention_Origine", None))
            niveau = row.get("Niveau", None)
            source = row.get("Source", None)

            champs = []
            for col in df.columns:
                if col in COLONNES_METADONNEES:
                    continue
                val = row[col]
                if pd.notna(val) and str(val).strip() != "":
                    champs.append(f"{col}: {val}")

            if not champs:
                continue

            row_text = f"FEUILLE: {sheet_name} | " + " | ".join(champs)
            citation_ref = f"[Feuille: '{sheet_name}' | Ligne {index + 2}]"

            chunks.append({
                "id": len(chunks),
                "text": row_text,
                "citation": citation_ref,
                "sheet": sheet_name,
                "row_idx": index + 2,
                "code_parcours": code_parcours if pd.notna(code_parcours) else None,
                "mention": str(mention) if pd.notna(mention) else None,
                "niveau": str(niveau) if pd.notna(niveau) else None,
                "source_officielle": str(source) if pd.notna(source) else None,
                "type": "ligne",
            })
    return chunks


def construire_chunks_synthese_parcours(sheets_dict):
    """
    Crée, pour chaque parcours, un chunk de synthèse regroupant description, matières tous
    niveaux confondus, compétences et débouchés. Évite au moteur de recherche de devoir
    combiner 5-6 chunks séparés pour répondre à une question globale du type
    "que fait-on dans le parcours IGGLIA ?".
    """
    df_form = sheets_dict[SHEET_FORMATIONS]
    df_comp = sheets_dict[SHEET_COMPETENCES]
    df_deb = sheets_dict[SHEET_DEBOUCHES]
    df_rel = sheets_dict[SHEET_RELATION]

    chunks = []
    for code in sorted(df_form["Code_Parcours"].dropna().unique()):
        lignes = df_form[df_form["Code_Parcours"] == code].sort_values("Niveau")
        if lignes.empty:
            continue

        premiere = lignes.iloc[0]
        mention = premiere.get("Mention")
        nom_parcours = premiere.get("Nom_Parcours")
        description = premiere.get("Description_Parcours")

        matieres_par_niveau = []
        for _, l in lignes.iterrows():
            if pd.notna(l.get("Matière")):
                matieres_par_niveau.append(f"Niveau {l.get('Niveau')} ({l.get('Diplôme')}): {l.get('Matière')}")

        comp_row = df_comp[df_comp["Code_Parcours"] == code]
        competences_dev = comp_row.iloc[0].get("Compétences développées") if not comp_row.empty else None

        deb_row = df_deb[df_deb["Code_Parcours"] == code]
        metiers = deb_row.iloc[0].get("Metiers_Estimes") if not deb_row.empty else None

        rel_row = df_rel[df_rel["Code_Parcours"] == code]
        matieres_pivot = rel_row.iloc[0].get("Matieres_Pivot") if not rel_row.empty else None
        comp_tech = rel_row.iloc[0].get("Competences_Techniques") if not rel_row.empty else None
        comp_trans = rel_row.iloc[0].get("Competences_Transversales") if not rel_row.empty else None

        parties = [f"SYNTHÈSE PARCOURS {code} ({nom_parcours}, mention {mention})"]
        if pd.notna(description):
            parties.append(f"Description: {description}")
        if matieres_par_niveau:
            parties.append("Programme par niveau: " + " ; ".join(matieres_par_niveau))
        if pd.notna(competences_dev):
            parties.append(f"Compétences développées: {competences_dev}")
        if pd.notna(matieres_pivot):
            parties.append(f"Matières pivot: {matieres_pivot}")
        if pd.notna(comp_tech):
            parties.append(f"Compétences techniques: {comp_tech}")
        if pd.notna(comp_trans):
            parties.append(f"Compétences transversales: {comp_trans}")
        if pd.notna(metiers):
            parties.append(f"Débouchés / métiers estimés: {metiers}")

        chunks.append({
            "id": None,
            "text": " | ".join(parties),
            "citation": f"[Synthèse consolidée du parcours {code}]",
            "sheet": "synthese",
            "row_idx": None,
            "code_parcours": code,
            "mention": str(mention) if pd.notna(mention) else None,
            "niveau": None,
            "source_officielle": None,
            "type": "synthese",
        })
    return chunks


chunks = construire_chunks_lignes(sheets_dict) + construire_chunks_synthese_parcours(sheets_dict)
for i, c in enumerate(chunks):
    c["id"] = i

print(f"✅ {len(chunks)} chunks créés "
      f"({sum(1 for c in chunks if c['type']=='ligne')} lignes + "
      f"{sum(1 for c in chunks if c['type']=='synthese')} synthèses par parcours).")
if chunks:
    print("Exemple :", chunks[0]['citation'], "\n", chunks[0]['text'][:200], "...")


✅ 196 chunks créés (180 lignes + 16 synthèses par parcours).
Exemple : [Feuille: '01_Formations_Matieres' | Ligne 2] 
 FEUILLE: 01_Formations_Matieres | ID_Formation: INF-IGGLIA | Mention: INFORMATIQUE ET TELECOMMUNICATION | Code_Parcours: IGGLIA | Nom_Parcours: IGGLIA | Niveau: 1 | Diplôme: Bacc +1 | Description_Parc ...


## 4. Pipeline de recherche hybride (vectorielle + BM25) avec normalisation français

In [4]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from unidecode import unidecode
import faiss
import numpy as np

STOPWORDS_FR = {
    "le", "la", "les", "de", "des", "du", "un", "une", "et", "en", "a", "au", "aux", "d", "l",
    "que", "qui", "quoi", "dont", "est", "sont", "pour", "dans", "sur", "par", "avec", "ce",
    "cette", "ces", "son", "sa", "ses", "je", "tu", "il", "elle", "nous", "vous", "ils", "elles",
    "quel", "quelle", "quels", "quelles", "y", "a", "se", "ne", "pas",
}

def tokenize_fr(text):
    """Tokenisation robuste : minuscules, accents neutralisés, ponctuation supprimée, mots-outils filtrés."""
    text_clean = unidecode(text.lower())
    text_clean = re.sub(r'[^\w\s]', ' ', text_clean)
    return [w for w in text_clean.split() if len(w) > 1 and w not in STOPWORDS_FR]

tokenized_corpus = [tokenize_fr(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print("Encodage des embeddings multilingues...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
corpus_embeddings = embedding_model.encode([c["text"] for c in chunks], show_progress_bar=True).astype("float32")

faiss.normalize_L2(corpus_embeddings)
dimension = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(corpus_embeddings)


def recherche_hybride(query, top_k=5, k_rrf=60, code_parcours_filtre=None):
    """
    Recherche hybride BM25 + FAISS avec fusion RRF (Reciprocal Rank Fusion).
    Un filtre optionnel sur code_parcours resserre la recherche quand le parcours visé
    est déjà connu (cf. détection d'intention plus bas) : les candidats hors parcours sont
    écartés, sauf si cela viderait complètement les résultats (repli automatique).
    """
    token_query = tokenize_fr(query)
    bm25_scores = bm25.get_scores(token_query)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k * 4]

    q_vector = embedding_model.encode([query]).astype("float32")
    faiss.normalize_L2(q_vector)
    _, faiss_top_indices = faiss_index.search(q_vector, top_k * 4)
    faiss_top_indices = faiss_top_indices[0]

    rrf_scores = {}
    for rank, idx in enumerate(bm25_top_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k_rrf + rank + 1))
    for rank, idx in enumerate(faiss_top_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k_rrf + rank + 1))

    candidats = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)

    if code_parcours_filtre:
        code_norm = normaliser_code(code_parcours_filtre)
        candidats_filtres = [idx for idx in candidats if chunks[idx].get("code_parcours") == code_norm]
        if candidats_filtres:
            candidats = candidats_filtres

    sorted_indices = candidats[:top_k]

    results = []
    for idx in sorted_indices:
        results.append({"chunk": chunks[idx], "score_rrf": rrf_scores.get(idx, 0.0)})
    return results

print("✅ Index Hybride BM25 + FAISS avec reranking RRF opérationnel !")


Encodage des embeddings multilingues...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Index Hybride BM25 + FAISS avec reranking RRF opérationnel !


## 5. Outils métiers (corrigés et enrichis)

In [5]:
import json

def lister_toutes_les_filieres():
    """Liste complète et unique de toutes les filières/parcours du dataset."""
    df = sheets_dict[SHEET_FORMATIONS]
    filieres = df[["Code_Parcours", "Nom_Parcours", "Mention", "Niveau"]].drop_duplicates(subset=["Code_Parcours"])
    resultats = []
    for idx, row in filieres.iterrows():
        resultats.append({
            "source": f"(Source : Offre de formation - Ligne {idx + 2})",
            "code": row.get("Code_Parcours"),
            "nom": row.get("Nom_Parcours"),
            "mention": row.get("Mention"),
        })
    return resultats


def rechercher_formation(query=None, code_parcours=None, mention=None, niveau=None):
    """Recherche des formations en combinant filtres structurés et recherche hybride RAG."""
    df = sheets_dict[SHEET_FORMATIONS]
    filtered = df.copy()

    if code_parcours:
        filtered = filtered[filtered["Code_Parcours"] == normaliser_code(code_parcours)]
    if mention:
        filtered = filtered[filtered["Mention"].astype(str).str.upper().str.contains(mention.upper(), na=False)]
    if niveau is not None:
        try:
            niveau_int = int(float(niveau))
            filtered = filtered[filtered["Niveau"] == niveau_int]
        except (ValueError, TypeError):
            pass

    results = []
    for idx, row in filtered.iterrows():
        results.append({
            "citation": f"[Feuille: '{SHEET_FORMATIONS}' | Ligne {idx + 2}]",
            "Code_Parcours": row.get("Code_Parcours"),
            "Nom_Parcours": row.get("Nom_Parcours"),
            "Niveau": row.get("Niveau"),
            "Diplôme": row.get("Diplôme"),
            "Matières": row.get("Matière"),
            "Description": row.get("Description_Parcours"),
        })

    if query and len(results) == 0:
        hyb_res = recherche_hybride(query, top_k=3, code_parcours_filtre=code_parcours)
        return [{"citation": r["chunk"]["citation"], "details": r["chunk"]["text"]} for r in hyb_res]

    return results


def obtenir_matieres_pivot_et_competences(code_parcours):
    """
    Récupère, pour un parcours donné, les matières pivot et compétences (techniques et
    transversales) issues du référentiel de correspondance parcours/métiers/compétences.
    Remplace l'ancien outil 'verifier_prerequis' qui pointait vers une feuille inexistante.
    """
    code = normaliser_code(code_parcours)
    df = sheets_dict[SHEET_RELATION]
    rows = df[df["Code_Parcours"] == code]

    results = []
    for idx, row in rows.iterrows():
        results.append({
            "citation": f"[Feuille: '{SHEET_RELATION}' | Ligne {idx + 2}]",
            "Matieres_Pivot": row.get("Matieres_Pivot"),
            "Competences_Techniques": row.get("Competences_Techniques"),
            "Competences_Transversales": row.get("Competences_Transversales"),
        })
    return results


def obtenir_conditions_acces_l1():
    """
    Renvoie les conditions d'accès en L1 (pièces à fournir, frais) ainsi que la liste des
    départements de l'établissement, telles que collectées dans la feuille dédiée.
    """
    df = sheets_dict[SHEET_CONDITIONS]
    categories_connues = {"Sélection", "Frais d'inscription", "Frais de scolarité"}

    conditions, departements = [], []
    for idx, row in df.iterrows():
        cat = row.get("Catégorie")
        if pd.isna(cat):
            continue
        if cat in categories_connues:
            conditions.append({
                "citation": f"[Feuille: '{SHEET_CONDITIONS}' | Ligne {idx + 2}]",
                "categorie": cat,
                "element": row.get("Élément / Document requis") if pd.notna(row.get("Élément / Document requis")) else None,
                "destinataire": row.get("Destinataire / Précision") if pd.notna(row.get("Destinataire / Précision")) else None,
                "tarif": row.get("Tarif / Droit (Ar)") if pd.notna(row.get("Tarif / Droit (Ar)")) else None,
                "remarques": row.get("Remarques") if pd.notna(row.get("Remarques")) else None,
            })
        elif str(cat).strip().lower() != "département":
            departements.append(str(cat))
    return {"conditions": conditions, "departements": departements}


def calculer_score_adequation(competences_profil, code_parcours):
    """
    Calcule le score de correspondance (%) entre un profil de compétences/intérêts et un
    parcours, en croisant la feuille 'Compétences développées' ET la feuille 'Relation'
    (matières pivot + compétences techniques/transversales) pour une évaluation plus complète
    qu'un simple matching sur une seule colonne.
    """
    code = normaliser_code(code_parcours)
    df_comp = sheets_dict[SHEET_COMPETENCES]
    df_rel = sheets_dict[SHEET_RELATION]

    row_comp = df_comp[df_comp["Code_Parcours"] == code]
    row_rel = df_rel[df_rel["Code_Parcours"] == code]

    if row_comp.empty and row_rel.empty:
        return {"score_adequation": "0%", "remarque": f"Parcours {code_parcours} non trouvé."}

    texte_reference = ""
    citations = []
    if not row_comp.empty:
        idx = row_comp.index[0]
        texte_reference += " " + str(row_comp.iloc[0].get("Compétences développées", ""))
        citations.append(f"[Feuille: '{SHEET_COMPETENCES}' | Ligne {idx + 2}]")
    if not row_rel.empty:
        idx = row_rel.index[0]
        r = row_rel.iloc[0]
        texte_reference += " " + " ".join(
            str(r.get(c, "")) for c in ["Matieres_Pivot", "Competences_Techniques", "Competences_Transversales"]
        )
        citations.append(f"[Feuille: '{SHEET_RELATION}' | Ligne {idx + 2}]")

    texte_reference = unidecode(texte_reference.lower())
    matches = [c for c in competences_profil if unidecode(c.lower()) in texte_reference]
    score_pct = round((len(matches) / max(len(competences_profil), 1)) * 100, 1)

    return {
        "citation": citations,
        "code_parcours": code,
        "score_adequation": f"{score_pct}%",
        "competences_validees": matches,
    }


def identifier_debouches_et_passerelles(code_parcours):
    """Récupère les débouchés métiers estimés et les passerelles possibles d'un parcours."""
    code = normaliser_code(code_parcours)
    df_deb = sheets_dict[SHEET_DEBOUCHES]
    df_pas = sheets_dict[SHEET_PASSERELLES]

    row_deb = df_deb[df_deb["Code_Parcours"] == code]
    rows_pas = df_pas[df_pas["Code_Parcours_Origine"] == code]

    debouches = []
    for idx, r in row_deb.iterrows():
        debouches.append({
            "citation": f"[Feuille: '{SHEET_DEBOUCHES}' | Ligne {idx + 2}]",
            "Metiers_Estimes": r.get("Metiers_Estimes"),
        })

    passerelles = []
    for idx, r in rows_pas.iterrows():
        passerelles.append({
            "citation": f"[Feuille: '{SHEET_PASSERELLES}' | Ligne {idx + 2}]",
            "Code_Parcours_Cible": r.get("Code_Parcours_Cible"),
            "Type_Passerelle": r.get("Type_Passerelle"),
            "Justification": r.get("Justification"),
        })

    return {"debouches": debouches, "passerelles": passerelles}


print("✅ Outils métiers initialisés : recherche formation, matières pivot/compétences, "
      "conditions d'accès, score d'adéquation, débouchés/passerelles, liste complète.")


✅ Outils métiers initialisés : recherche formation, matières pivot/compétences, conditions d'accès, score d'adéquation, débouchés/passerelles, liste complète.


## 6. Détection légère d'intention et de parcours cité dans la question

In [6]:
CODES_CONNUS = sorted(
    sheets_dict[SHEET_FORMATIONS]["Code_Parcours"].dropna().unique().tolist(),
    key=len, reverse=True,
)

def detecter_code_parcours(question):
    """Repère si un code de parcours connu (ex: IGGLIA, FIC, CAA...) est cité dans la question."""
    q_norm = unidecode(question.upper())
    for code in CODES_CONNUS:
        if re.search(rf"\b{re.escape(code)}\b", q_norm):
            return code
    return None


MOTS_CLES_INTENTIONS = {
    "conditions_acces": ["condition", "inscription", "acces", "frais", "droit d'inscription",
                          "bac", "piece", "dossier", "tarif"],
    "debouches_passerelles": ["debouche", "metier", "carriere", "travailler", "passerelle",
                               "reorientation", "changer de parcours"],
    "matieres_competences": ["matiere", "programme", "cours", "competence", "pivot"],
    "liste_complete": ["toutes les filieres", "tous les filieres", "tout les filieres",
                        "tous les parcours", "tout les parcours", "toutes les parcours",
                        "toutes les formations", "liste des filieres", "liste des formations",
                        "liste des parcours"],
}

def detecter_intentions(question):
    """Détecte, par mots-clés (accents neutralisés), les intentions présentes dans la question."""
    q_norm = unidecode(question.lower())
    intentions = set()
    for intention, mots in MOTS_CLES_INTENTIONS.items():
        if any(unidecode(m) in q_norm for m in mots):
            intentions.add(intention)
    return intentions

print("✅ Détection d'intention et de code parcours prête.")
print("Exemple :", detecter_code_parcours("Quels sont les débouchés du parcours IGGLIA ?"),
      detecter_intentions("Quels sont les débouchés et les passerelles du parcours IGGLIA ?"))


✅ Détection d'intention et de code parcours prête.
Exemple : IGGLIA {'debouches_passerelles', 'matieres_competences'}


## 7. Configuration LLM & Agent RAG avec routage automatique vers les outils

In [14]:
from huggingface_hub import InferenceClient

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("RAGORIENTIA_TOKEN")  # ⚠️ nom du secret corrigé (était "HFY_TOKEN" dans la version d'origine)
except ImportError:
    import getpass
    HF_TOKEN = os.environ.get("RAGORIENTIA_TOKEN") or getpass.getpass("Entrez votre HF_TOKEN : ")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN introuvable. Sur Colab : Outils > Secrets, ajoutez un secret nommé 'HF_TOKEN'. "
        "En local : définissez la variable d'environnement HF_TOKEN."
    )

client = InferenceClient(api_key=HF_TOKEN)

def call_llm(prompt):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=750,
        temperature=0.2,
    )
    return response.choices[0].message.content


def ask_rag_agent(question, code_parcours_cible=None):
    """
    Orchestre : détection d'intention -> appel des outils métiers pertinents -> recherche
    hybride RAG (resserrée sur le parcours détecté si possible) -> génération de la réponse
    avec citations obligatoires.
    """
    code_detecte = code_parcours_cible or detecter_code_parcours(question)
    intentions = detecter_intentions(question)

    blocs_donnees = []

    # 1. Cas particulier : demande d'une liste exhaustive de filières
    if "liste_complete" in intentions:
        donnees = lister_toutes_les_filieres()
        blocs_donnees.append(f"[DONNÉES EXHAUSTIVES DES FILIÈRES]: {json.dumps(donnees, ensure_ascii=False)}")

    # 2. Recherche hybride RAG
    top_k = 3 if (code_detecte and "liste_complete" not in intentions) else 5
    rag_results = recherche_hybride(question, top_k=top_k, code_parcours_filtre=code_detecte)
    for res in rag_results:
        c = res["chunk"]
        blocs_donnees.append(f"{c['citation']} {c['text']}")

    # 3. Outils métiers déclenchés selon l'intention détectée et le parcours identifié
    if code_detecte:
        if "debouches_passerelles" in intentions or not intentions:
            deb_pas = identifier_debouches_et_passerelles(code_detecte)
            blocs_donnees.append(f"[OUTIL DÉBOUCHÉS/PASSERELLES - {code_detecte}]: {json.dumps(deb_pas, ensure_ascii=False)}")
        if "matieres_competences" in intentions:
            comp = obtenir_matieres_pivot_et_competences(code_detecte)
            blocs_donnees.append(f"[OUTIL MATIÈRES PIVOT/COMPÉTENCES - {code_detecte}]: {json.dumps(comp, ensure_ascii=False)}")

    if "conditions_acces" in intentions:
        conditions = obtenir_conditions_acces_l1()
        blocs_donnees.append(f"[OUTIL CONDITIONS D'ACCÈS EN L1]: {json.dumps(conditions, ensure_ascii=False)}")

    contexte_total = "\n---\n".join(blocs_donnees)

    prompt = f"""Tu es ORIENT'IA, un assistant expert en orientation académique et professionnelle.

Consignes de rédaction :
1. Rédige une réponse claire, synthétique, fluide et directement compréhensible.
2. Structure la réponse avec soin (introduction directe, points clés à puces, synthèses et débouchés si pertinent).
3. Traçabilité des sources (VRAIES VALEURS DE SOURCE) :
   - N'affiche PAS de balises techniques brutes de type `[Feuille: '...' | Ligne X]`.
   - À la place, mentionne explicitement la VRAIE SOURCE des données de manière naturelle et lisible (ex: `(Source : Passerelles entre formations)`, `(Source : Programme & Matières IGGLIA)`, `(Source : Référentiel des compétences)`).
   - Indique la source réelle associée directement après l'information concernée.
   - À la fin de ton explication, ajoute une section "Sources des données" récapitulant clairement les vraies sources d'information consultées pour cette réponse.
4. Base-toi exclusivement sur les données collectées ci-dessous sans halluciner d'informations. Si une information demandée n'est pas dans les données, dis-le explicitement plutôt que de l'inventer.

CONTEXTE ET DONNÉES COLLECTÉES :
{contexte_total}

QUESTION DE L'UTILISATEUR :
{question}

RÉPONSE CLAIRE ET PRÉCISE :"""
    return call_llm_groq(prompt)

print("✅ Agent RAG avec routage automatique des outils prêt.")


✅ Agent RAG avec routage automatique des outils prêt.


## 8. Démonstration des outils & tests du RAG

In [8]:
print("=== TEST OUTIL : Rechercher Formation (IGGLIA Niveau 1) ===")
print(rechercher_formation(code_parcours="IGGLIA", niveau=1))

print("\n=== TEST OUTIL : Score d'Adéquation ===")
profil = ["programmation", "bases de données", "intelligence artificielle", "statistiques"]
print(calculer_score_adequation(profil, "IGGLIA"))

print("\n=== TEST OUTIL : Débouchés & Passerelles ===")
print(identifier_debouches_et_passerelles("IGGLIA"))

print("\n=== TEST OUTIL : Matières pivot & compétences ===")
print(obtenir_matieres_pivot_et_competences("ISAIA"))

print("\n=== TEST OUTIL : Conditions d'accès en L1 ===")
print(obtenir_conditions_acces_l1())


=== TEST OUTIL : Rechercher Formation (IGGLIA Niveau 1) ===
[{'citation': "[Feuille: '01_Formations_Matieres' | Ligne 2]", 'Code_Parcours': 'IGGLIA', 'Nom_Parcours': 'IGGLIA', 'Niveau': 1, 'Diplôme': 'Bacc +1', 'Matières': 'Algèbre, Algorithmes, Analyse, Bases de données, Comptabilité, Français, HTML/CSS, Informatique scientifique, Mathématique Discrètes, Mathématique Financière, Organisation, PASCAL, Probabilités- Statistiques, Structures de données', 'Description': "Toutes les entreprises (publiques ou privées) ne peuvent plus se passer de l'outil informatique surtout l'informatique appliquée à la gestion. La filière Informatique de Gestion Génie Logiciel et Intelligence Artificielle est une filière dont l'objectif est la formation d'Ingénieurs et de Techniciens Supérieurs capables de maîtriser toutes les techniques informatiques relatives à la gestion des entreprises."}]

=== TEST OUTIL : Score d'Adéquation ===
{'citation': ["[Feuille: 'Compétences développées' | Ligne 2]", "[Feuill

In [11]:
question1 = "Donne moi la liste de tous les parcours"
print("Q1:", question1)
print(ask_rag_agent(question1))

print("\n" + "=" * 80 + "\n")




Q1: Donne moi la liste de tous les parcours
**Liste des parcours proposés (codes + noms)**  

| Code | Nom du parcours | Mention | Source |
|------|-----------------|---------|--------|
| IGGLIA | IGGLIA | INFORMATIQUE ET TELECOMMUNICATION | (Source : Offre de formation – Ligne 2) |
| ISAIA | ISAIA | INFORMATIQUE ET TELECOMMUNICATION | (Source : Offre de formation – Ligne 7) |
| CAA | CAA | TECHNIQUES DES AFFAIRES | (Source : Offre de formation – Ligne 12) |
| FIC | FIC | TECHNIQUES DES AFFAIRES | (Source : Offre de formation – Ligne 17) |
| IAA | IAA | BIOTECHNOLOGIE ET AGRONOMIE | (Source : Offre de formation – Ligne 22) |
| PIP | PIP | BIOTECHNOLOGIE ET AGRONOMIE | (Source : Offre de formation – Ligne 27) |
| AEE | AEE | BIOTECHNOLOGIE ET AGRONOMIE | (Source : Offre de formation – Ligne 32) |
| EMII | EMII | GENIE INDUSTRIEL ET GENIE CIVIL | (Source : Offre de formation – Ligne 37) |
| GCA | GCA | GENIE INDUSTRIEL ET GENIE CIVIL | (Source : Offre de formation – Ligne 42) |
| TEE | T

## 9. Variante alternative : servir le LLM via Groq (gratuit, hors quota Hugging Face)

Si le quota gratuit d'Inference Providers de Hugging Face reste trop juste même avec `gpt-oss-20b`, Groq héberge
directement le **même modèle** (`openai/gpt-oss-20b`, ainsi que `openai/gpt-oss-120b`) avec des limites de débit
gratuites généreuses et totalement indépendantes des crédits Hugging Face.

Le reste du pipeline (chunking, recherche hybride, outils métiers, `ask_rag_agent`) ne change pas : seule la
fonction `call_llm` change de backend. Il suffit de créer une clé gratuite sur
[console.groq.com/keys](https://console.groq.com/keys).


In [12]:
!pip install -q groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.4 MB/s eta 0:00:00


In [13]:
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_KEY")
except ImportError:
    import getpass
    GROQ_API_KEY = os.environ.get("GROQ_KEY") or getpass.getpass("Entrez votre GROQ_API_KEY : ")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY introuvable. Créez une clé gratuite sur https://console.groq.com/keys, "
        "puis sur Colab : Outils > Secrets, ajoutez un secret nommé 'GROQ_API_KEY'."
    )

groq_client = Groq(api_key=GROQ_API_KEY)


def call_llm_groq(prompt):
    """Variante de call_llm utilisant Groq au lieu des Inference Providers Hugging Face."""
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",  # même modèle que la cellule LLM principale, servi par Groq
        messages=[{"role": "user", "content": prompt}],
        max_tokens=750,
        temperature=0.2,
    )
    return response.choices[0].message.content


# Pour basculer tout le pipeline RAG (ask_rag_agent, etc.) sur Groq sans rien modifier
# d'autre dans le notebook, il suffit de réassigner la fonction utilisée par l'agent :
call_llm = call_llm_groq

print("✅ Client Groq initialisé (modèle : openai/gpt-oss-20b). "
      "L'agent ask_rag_agent utilise maintenant Groq au lieu de Hugging Face.")


✅ Client Groq initialisé (modèle : openai/gpt-oss-20b). L'agent ask_rag_agent utilise maintenant Groq au lieu de Hugging Face.


In [30]:
# Test rapide de la variante Groq
question_test = "Quel est le nombre de tous les filières à l'ISPM?"
print(ask_rag_agent(question_test))


**Nombre total de filières à l’ISPM : 16**

- IGGLIA, ISAIA, CAA, FIC, IAA, PIP, AEE, EMII, GCA, TEE, TEH, ESIIA, IMTICIA, DTJA, EMP, ICMP  
  *(Source : Offre de formation – lignes 2, 7, 12, 17, 22, 27, 32, 37, 42, 47, 52, 57, 62, 67, 72, 77)*  

---

### Sources des données
- Offre de formation – lignes 2, 7, 12, 17, 22, 27, 32, 37, 42, 47, 52, 57, 62, 67, 72, 77.
